In [76]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers

In [77]:
import scipy.optimize

verbose = False

delta = 2
m = 2

In [78]:
def test_for_srns(sample: behaviors.RoutedBehavior):
    srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
        delta=delta,
        m=m,
        measured_behavior=sample,
        )

    A_eq, b_eq = srns_set.get_equations(measured_behavior=sample)

    q_shape = behaviors.LatentSRNSBehavior(
        delta=delta,
        m=m,
    ).vector_shape[0]

    lb = np.zeros(q_shape+1)
    rb = np.ones(q_shape+1)
    rb[0] = 2

    bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]

    c = np.zeros(q_shape+1)
    c[0] = -1

    res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
        c=c,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
    )

    return res


In [79]:
sampler = samplers.NoSignalingSampler(delta, m)

In [80]:
sampled_behavior = sampler.sample()
print(f"Sampled behavior is tested [{sampled_behavior.is_no_signaling()}] to being no signaling")

print()

res = test_for_srns(sampled_behavior)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

2025-05-20 10:55:16.765 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)


2025-05-20 10:55:16.767 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


Sampled behavior is tested [True] to being no signaling

True
1.0077938894137157
0
[1.00779389 0.08496747 0.46166584 0.17427914 0.43089333 0.39593676
 0.0192384  0.30971929 0.0531051  0.51909577 0.08331631 0.4297841
 0.11408882 0.         0.43577945 0.08621747 0.40191275 0.
 0.00509756 0.07575098 0.         0.06358715 0.02138454 0.34156611
 0.45751633 0.00509756 0.         0.0605835  0.13633447 0.03035892
 0.07256153 0.42305579 0.30710556]


## Post-analysis

In [81]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(res.x[1:]), 0, 1),
)

test_tolerance = 1e-10

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling(atol=test_tolerance)}] to being no signaling")


Yielded behavior is Behavior:
Short path (z=S):
[[0.08496747 0.46166584 0.17427914 0.43089333]
 [0.39593676 0.0192384  0.30971929 0.0531051 ]
 [0.51909577 0.08331631 0.4297841  0.11408882]
 [0.         0.43577945 0.08621747 0.40191275]]
Long path (z=L) :
[[0.         0.00509756]
 [0.07575098 0.        ]
 [0.06358715 0.02138454]
 [0.34156611 0.45751633]
 [0.00509756 0.        ]
 [0.0605835  0.13633447]
 [0.03035892 0.07256153]
 [0.42305579 0.30710556]]
------------
Yielded behavior is tested [True] to being no signaling


In [82]:
# sanity_check = A_eq[:, 1:] @ yielded_behavior.get_vector()
# sanity_check = sanity_check[:2*delta**2*m**2]

In [83]:
# sanity_check = behaviors.RoutedBehavior(
#     delta=delta,
#     m=m,
#     vector=sanity_check,
# )
# print(f"Sanity check is {sanity_check}")
# print(f"Sanity check is tested [{sanity_check.is_no_signaling()}] to being no signaling")

In [84]:
# res = test_for_srns(sampled_behavior)

# print(res.success)
# print(-res.fun)
# print(res.status)
# print(res.x)

In [85]:
# res = test_for_srns(behaviors.pr_box)

# print(res.success)
# print(-res.fun)
# print(res.status)
# print(res.x)

# pr_match = behaviors.LatentSRNSBehavior(
#     delta=delta,
#     m=m,
#     vector=np.clip(np.array(res.x[1:]), 0, 1),
# )
# print(f"PR match is {pr_match}")
# print(f"PR match is tested [{pr_match.is_no_signaling()}] to being no signaling")

# pr_box_check = A_eq[:, 1:] @ pr_match.get_vector()
# pr_box_check = pr_box_check[:2*delta**2*m**2]
# pr_box_check = behaviors.RoutedBehavior(
#     delta=delta,
#     m=m,
#     vector=pr_box_check,
# )
# print(f"PR box check is {pr_box_check}")
# print(f"PR box check is tested [{pr_box_check.is_no_signaling()}] to being no signaling")